# Build a Forecasting Model

In the EDA notebook we got a feel for the Walmart sales data. Now we put
that understanding to work and build a **weekly sales forecast** at the
`(Store, Department)` level.

We will go through the same workflow you would follow on a real
forecasting project:

1. **Load and merge** the source files.
2. **Engineer features** — calendar parts, lag and rolling features, store
   attributes.
3. **Split** the data using a **time-based** holdout (no random shuffles).
4. **Build a baseline** so we know what "good" looks like.
5. **Train a linear regression** as a simple, interpretable benchmark.
6. **Train a gradient-boosted tree (XGBoost)** model.
7. **Evaluate** with MAE, RMSE, and the competition's WMAE.
8. **Interpret** the results and discuss how to improve them.


## Setup


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import xgboost as xgb

print("pandas  :", pd.__version__)
print("numpy   :", np.__version__)
print("xgboost :", xgb.__version__)

## Load and Merge the Data

We pull the same three files we used in EDA and join them into a single
modeling table.


In [ ]:
BASE = "https://raw.githubusercontent.com/bdi593/datasets/refs/heads/main/walmart-store-sales-forecasting"

df_train    = pd.read_csv(f"{BASE}/train.csv",    parse_dates=["Date"])
df_features = pd.read_csv(f"{BASE}/features.csv", parse_dates=["Date"])
df_stores   = pd.read_csv(f"{BASE}/stores.csv")

df = (
    df_train
    .merge(df_features, on=["Store", "Date", "IsHoliday"], how="left")
    .merge(df_stores,   on="Store", how="left")
    .sort_values(["Store", "Dept", "Date"])
    .reset_index(drop=True)
)

print("rows:", len(df), "  cols:", df.shape[1])
df.head()

## Feature Engineering

The two biggest gains in retail forecasting almost always come from:

1. **Calendar features** that let the model learn seasonality.
2. **Lag and rolling features** that let the model use recent history of
   the *same* `(Store, Dept)` series as a predictor.

We also keep store **type** and **size** as static features.


### Calendar features


In [ ]:
df["Year"]       = df["Date"].dt.year
df["Month"]      = df["Date"].dt.month
df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)
df["IsHoliday"]  = df["IsHoliday"].astype(int)

# Rough flags for the four key holidays (dates given by the dataset).
super_bowl   = ["2010-02-12", "2011-02-11", "2012-02-10"]
labor_day    = ["2010-09-10", "2011-09-09", "2012-09-07"]
thanksgiving = ["2010-11-26", "2011-11-25"]
christmas    = ["2010-12-31", "2011-12-30"]

df["IsSuperBowl"]   = df["Date"].isin(pd.to_datetime(super_bowl)).astype(int)
df["IsLaborDay"]    = df["Date"].isin(pd.to_datetime(labor_day)).astype(int)
df["IsThanksgiving"]= df["Date"].isin(pd.to_datetime(thanksgiving)).astype(int)
df["IsChristmas"]   = df["Date"].isin(pd.to_datetime(christmas)).astype(int)

### Lag and rolling features

For each `(Store, Dept)` series we add:

- the **lag** of `Weekly_Sales` from 1 week ago and from 52 weeks ago
  (last year's same week — a natural seasonal benchmark), and
- the **rolling mean** of the last 4 weeks.

We compute them with `groupby(...).shift(...)` so that no future
information ever leaks into a row's features.


In [ ]:
grp = df.groupby(["Store", "Dept"])["Weekly_Sales"]

df["Sales_Lag1"]    = grp.shift(1)
df["Sales_Lag52"]   = grp.shift(52)
df["Sales_Roll4"]   = grp.shift(1).rolling(4).mean().reset_index(level=[0, 1], drop=True)

The first few weeks of each series have no history, so the lag features
are `NaN` there. Rather than drop those rows entirely we fill the missing
values with the per-`(Store, Dept)` median — a simple imputation that
keeps the early weeks available for training.


In [ ]:
for col in ["Sales_Lag1", "Sales_Lag52", "Sales_Roll4"]:
    df[col] = df[col].fillna(
        df.groupby(["Store", "Dept"])[col].transform("median")
    )
    df[col] = df[col].fillna(df[col].median())  # safety net for new (Store, Dept)

# Markdown columns: missing means "no markdown that week".
markdown_cols = [f"MarkDown{i}" for i in range(1, 6)]
df[markdown_cols] = df[markdown_cols].fillna(0)

# CPI / Unemployment: a tiny number missing late in the series — forward fill.
df[["CPI", "Unemployment"]] = (
    df.groupby("Store")[["CPI", "Unemployment"]].ffill().bfill()
)

df.isna().sum().sort_values(ascending=False).head()

## Train / Test Split

This is the most important step in any forecasting project. **You must
split by time**, not at random. If we shuffle, future weeks leak into the
training set and we end up with optimistic, useless metrics.

We hold out the **last 12 weeks** of the dataset as our test set and
train on everything before.


In [ ]:
cutoff = df["Date"].max() - pd.Timedelta(weeks=12)
print("training through:", cutoff.date())

train = df[df["Date"] <= cutoff].copy()
test  = df[df["Date"] >  cutoff].copy()

print(f"train rows: {len(train):>8,}  ({train['Date'].min().date()} → {train['Date'].max().date()})")
print(f"test  rows: {len(test):>8,}  ({test['Date'].min().date()} → {test['Date'].max().date()})")

## A Sensible Baseline

Before training anything fancy we predict next week's sales as **the same
week from last year** (`Sales_Lag52`). This "seasonal naive" baseline is
hard to beat in retail because so much demand is annual.


In [ ]:
def regression_metrics(y_true, y_pred, weights=None):
    """Return MAE, RMSE, and (optionally) WMAE in a tidy dict."""
    out = {
        "MAE":  mean_absolute_error(y_true, y_pred),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
    }
    if weights is not None:
        weights = np.asarray(weights, dtype=float)
        out["WMAE"] = float(
            np.sum(weights * np.abs(np.asarray(y_true) - np.asarray(y_pred)))
            / weights.sum()
        )
    return out


# Holiday weeks are weighted 5x in the Walmart competition.
test_weights = np.where(test["IsHoliday"] == 1, 5.0, 1.0)

baseline_pred = test["Sales_Lag52"].values
baseline_metrics = regression_metrics(test["Weekly_Sales"], baseline_pred, weights=test_weights)
baseline_metrics

### How to read these numbers

- **MAE (Mean Absolute Error).** On average, the baseline is off by this
  many dollars per `(Store, Dept, Week)`. It is in the same units as the
  target, which makes it easy to talk to business stakeholders.
- **RMSE (Root Mean Squared Error).** Like MAE but penalizes large
  misses more heavily. If RMSE is much larger than MAE, a few big errors
  are dragging the score down.
- **WMAE (Weighted MAE).** Walmart's competition metric — identical to
  MAE except holiday weeks count 5×. It rewards models that nail the
  high-stakes weeks.

Any model we build needs to **beat these numbers** to be worth using.


## Feature Set

We separate the columns into:

- **`numeric_features`** — anything we can pass to a model directly.
- **`categorical_features`** — the store `Type`; we will one-hot encode it
  for the linear model and pass it as a category to XGBoost.


In [ ]:
numeric_features = [
    "Store", "Dept", "Size",
    "Temperature", "Fuel_Price", "CPI", "Unemployment",
    *[f"MarkDown{i}" for i in range(1, 6)],
    "Year", "Month", "WeekOfYear",
    "IsHoliday", "IsSuperBowl", "IsLaborDay", "IsThanksgiving", "IsChristmas",
    "Sales_Lag1", "Sales_Lag52", "Sales_Roll4",
]
categorical_features = ["Type"]
target = "Weekly_Sales"

X_train = train[numeric_features + categorical_features]
y_train = train[target]
X_test  = test[numeric_features + categorical_features]
y_test  = test[target]

## Linear Regression Benchmark

A regularized linear model (Ridge) is a great second baseline: it is
fast, interpretable, and tells us how much of the signal can be captured
by simple linear relationships.


In [ ]:
preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ],
    remainder="passthrough",
)

ridge = Pipeline([
    ("prep", preprocess),
    ("model", Ridge(alpha=1.0)),
])

ridge.fit(X_train, y_train)
ridge_pred = ridge.predict(X_test)

ridge_metrics = regression_metrics(y_test, ridge_pred, weights=test_weights)
ridge_metrics

## Gradient-Boosted Trees (XGBoost)

XGBoost is the workhorse of tabular forecasting. It handles non-linear
interactions, missing values, and large feature sets without much
hand-holding.


In [ ]:
X_train_xgb = X_train.copy()
X_test_xgb  = X_test.copy()
for col in categorical_features:
    X_train_xgb[col] = X_train_xgb[col].astype("category")
    X_test_xgb[col]  = X_test_xgb[col].astype("category")

model = xgb.XGBRegressor(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.9,
    colsample_bytree=0.9,
    enable_categorical=True,
    tree_method="hist",
    random_state=42,
)

model.fit(X_train_xgb, y_train)
xgb_pred = model.predict(X_test_xgb)

xgb_metrics = regression_metrics(y_test, xgb_pred, weights=test_weights)
xgb_metrics

## Compare the Models


In [ ]:
results = pd.DataFrame(
    {
        "Seasonal naive (lag-52)": baseline_metrics,
        "Ridge regression":        ridge_metrics,
        "XGBoost":                 xgb_metrics,
    }
).T.round(2)
results

### Interpreting the comparison

A few things to look for:

- **Did each model beat the seasonal naive baseline on every metric?** If
  not, the extra complexity isn't paying for itself.
- **Is WMAE much worse than MAE for any model?** That means the model
  struggles on holiday weeks — exactly the weeks the business cares about
  most.
- **Is RMSE noticeably bigger than MAE?** That points at a few large
  misses; investigating the worst predictions usually surfaces a missing
  feature (e.g., an unmodeled promotion).

In our experience XGBoost will beat Ridge by a wide margin here, mostly
because lag features interact with store and department in non-linear
ways that a linear model cannot capture.


## Feature Importance


In [ ]:
importances = (
    pd.DataFrame(
        {"feature": X_train_xgb.columns, "importance": model.feature_importances_}
    )
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

fig = px.bar(
    importances,
    x="importance",
    y="feature",
    orientation="h",
    title="XGBoost feature importance",
    height=600,
)
fig.update_yaxes(autorange="reversed")
fig.show()

importances.head(10)

**Takeaway.** Lag features (`Sales_Lag1`, `Sales_Lag52`, `Sales_Roll4`)
typically dominate, followed by store identity (`Store`, `Dept`, `Size`)
and the calendar features. That ordering matches the EDA: short-term
momentum and yearly seasonality are the two biggest sources of signal.


## Visualizing the Forecast

Looking at predictions on a single series helps build intuition for where
the model wins and where it loses.


In [ ]:
series = (
    test.assign(Pred=xgb_pred)
        .query("Store == 1 and Dept == 1")
        .sort_values("Date")
)

plot_df = series.melt(
    id_vars="Date",
    value_vars=["Weekly_Sales", "Pred"],
    var_name="Series",
    value_name="Sales",
)

fig = px.line(
    plot_df,
    x="Date",
    y="Sales",
    color="Series",
    markers=True,
    title="Store 1, Dept 1 — actual vs. XGBoost forecast (test window)",
)
fig.show()

### Residual Analysis

Where is the model wrong, and is it wrong in a structured way?


In [ ]:
resid = pd.DataFrame({
    "Date": test["Date"].values,
    "IsHoliday": test["IsHoliday"].values,
    "Residual": y_test.values - xgb_pred,
})

fig = px.box(
    resid,
    x="IsHoliday",
    y="Residual",
    title="Residuals: holiday vs. non-holiday weeks",
)
fig.show()

If the residuals are centered on zero and similarly spread on holiday vs.
non-holiday weeks, the model is well calibrated. A noticeable shift on
holidays would indicate the model is systematically under- or
over-predicting the weeks that matter most — a signal to add more holiday
features (e.g., weeks-to-Thanksgiving, post-Christmas indicators).


## Ideas to Improve the Model

What we built is already a respectable forecast, but every retail
forecasting project has more juice to squeeze. Things to try next:

1. **Richer calendar features.** Days to the next holiday, *post*-holiday
   indicators, paydays, school calendars, fiscal week of month.
2. **More lag and rolling features.** `Sales_Lag2`, `Sales_Lag4`,
   year-over-year growth, rolling standard deviation (a proxy for
   volatility).
3. **Hierarchical features.** Department-level chain totals or
   store-level chain totals as additional inputs to capture macro
   shifts.
4. **Better treatment of MarkDowns.** Indicator flags for "any markdown
   that week", interactions with department, or imputation that respects
   the post-2011 cutoff.
5. **Hyperparameter search.** Cross-validated tuning of `max_depth`,
   `learning_rate`, and `n_estimators` typically buys a meaningful WMAE
   improvement.
6. **Model-per-segment.** Train one model per `Type` (or even per
   department family) when behavior is very heterogeneous.
7. **Probabilistic forecasts.** Use quantile regression (`xgboost`'s
   `objective="reg:quantileerror"`) to produce prediction intervals,
   which inventory teams care about even more than point forecasts.
8. **Specialized libraries.** Try `sktime`, `statsforecast`, or `darts`
   for purpose-built forecasting workflows once you understand the basics.

:::{tip} Closing the loop with the business
A forecast is only valuable if it changes a decision. Before tuning
further, ask the inventory or merchandising team **what error tolerance
they need by store and department** — sometimes a 5% MAE improvement is
worth a quarter of effort, and sometimes it would not change the order
quantities at all.
:::
